# M14 — multi-seed equal-byte LoRanPAC confirmation (Kaggle, train-only)

Set **Accelerator = GPU** (P100 preferred) and **Internet = On**. Attach `zaphat206/cifar-100`. Create one private Dataset containing the exact M6, M11, and M13-N artifacts listed below, each renamed by appending `.bin` (do not recompress or change the bytes), then attach it. Use **Save Version → Save & Run All**. The run is long: 60 method/width/seed units are written atomically under `/kaggle/working`. A rerun in the same session reuses them. If Kaggle ends the session, attach the preceding notebook output as an Input on the next version; the run cell imports and validates those units before continuing. No test feature is created or read.

In [ ]:
REPO_GIT_URL='https://github.com/ZaPhat206/SOHO-CL.git'
REPO_COMMIT='2cf2093e407a54022d4d3e6de0ce6dd15a3cd1fe'
WORK_DIR='/kaggle/temp/SOHO-CL'
FEATURE_CACHE_DIR='/kaggle/temp/srq_m14_cifar_features'
OUTPUT_DIR='/kaggle/working/srq_m14_loranpac_output'
EXPORT_PATH='/kaggle/working/srq_generalization_m14_loranpac_multiseed_train_only.zip'
CONFIG='configs/srq_generalization_m14_loranpac_multiseed_train_only.json'
RUNNER='tools/srq_generalization_m14.py'
SOURCE_NAMES={'m6':'srq_generalization_m6_width_sweep_train_only.zip','m11':'srq_generalization_m11_adaptive_precision_train_only.zip','m13n':'srq_generalization_m13n_numerical_audit.zip'}
SOURCE_SHA={'m6':'b2739b9da023ebd2eedb6fdfe01c394e94f252773e847533b35350021c3d239e','m11':'65ce03df4da7041833014628b59aac1167f77bde2d64348b9a8a1e4fe09370a7','m13n':'726853486664cbf26ec109a061585a1effcd90569ce056108e9ff594f94e031d'}
CHECKPOINT_SHA='32aa17d6e17b43500f531d5f6dc9bc93e56ed8841b8a75682e1bb295d722405b'
CHECKPOINT_SIZE=346284714
BATCH_SIZE=128
NUM_WORKERS=2

In [ ]:
# Pinned checkout, dependencies, GPU, and portable source hashes.
import hashlib,json,os,shutil,subprocess,sys,zipfile
from pathlib import Path
assert Path('/kaggle/input').is_dir() and Path('/kaggle/working').is_dir()
os.chdir('/kaggle/working')
repo=Path(WORK_DIR); repo.parent.mkdir(parents=True,exist_ok=True)
if repo.exists(): shutil.rmtree(repo)
subprocess.run(['git','clone','--no-checkout',REPO_GIT_URL,WORK_DIR],check=True)
subprocess.run(['git','checkout','--detach',REPO_COMMIT],cwd=WORK_DIR,check=True)
os.chdir(WORK_DIR)
subprocess.run([sys.executable,'-m','pip','install','-q','-r','requirements-kaggle.txt','huggingface_hub'],check=True)
import torch
assert torch.cuda.is_available(),'Enable a Kaggle GPU and restart the session.'
def sha_raw(path): return hashlib.sha256(Path(path).read_bytes()).hexdigest()
def sha_source(path): return hashlib.sha256(Path(path).read_bytes().replace(b'\r\n',b'\n')).hexdigest()
EXPECTED={
 'configs/srq_generalization_m14_loranpac_multiseed_train_only.json':'562a697232f3a286f9eb517cadc68093b028087d3bdaaf85e0803a1923c3a9e7',
 'tools/srq_generalization_m14.py':'33359ee8dd53377099bb45070de363dfeb344c46e65ca203233ca79f3e6925be',
 'methods/frontends/loranpac.py':'b468d98981671876bbd223e62ab34ad8430d93f315662b0dabd36ab28ba980f7',
 'tools/srq_generalization_m12.py':'25c4d35bae98c7a324a4dc73e13a6eb91a6eb8e05290c58627093c76a9b98583',
 'tools/srq_generalization_m13.py':'a8b090ca4bf8a994e81222c63500fec79e3a1233065f416c886fd86cde24b82e',
 'tools/srq_generalization_m6.py':'bad119dca8b2c6e78200c81917c8e8b03a5c50923135f951d8723fd5afd2ae61',
 'tools/srq_generalization_m5.py':'4d08e27a825fb59d300ee5909542bca4bd550a8558bb137159a176f353739a84',
 'tools/srq_generalization_m4.py':'84302805f6c71475cfcd3f7c9f148700198e96879cac6c795ef0f1bbc0f4c29e',
 'tools/experiment_runner.py':'b2c953eea312a98ce4146757fb46a7dd0e4ebe20aa139da59464921b11f8310c',
 'models/backbone.py':'941e449dc6e66ca4018fb0d3ab3218d97ec97f498b557ed220c8332e75850a46',
 'utils/data_utils.py':'3cf85993e231b068ad5ae2f96be608b2e50e9c52f98fb2387fd3badfb44b6764',
 'utils/train_utils.py':'e24983bd3042ad82ec069916ba2853cf1c818cb2911ce193710c8ccd70e86bda',
 'tests/test_srq_generalization_m14.py':'02a615fd239b04864a157383ab7e88e414853bc52b27a975a7f0920bf7204b2c',
 'docs/research/SRQ_GENERALIZATION_M14_PLAN.md':'2c20217508519201613cc1476d8c69d46f2447a8efa6a0a1f353db56910b4ca3'}
for path,expected in EXPECTED.items(): assert sha_source(path)==expected,(path,sha_source(path),expected)
assert subprocess.check_output(['git','rev-parse','HEAD'],text=True).strip()==REPO_COMMIT
assert not subprocess.check_output(['git','status','--porcelain'],text=True).strip()
print('GPU:',torch.cuda.get_device_name(0),'| M14 SOURCE LOCK: PASS')

In [ ]:
# Discover byte-preserved source artifacts, raw CIFAR input, and checkpoint.
INPUT_ROOT=Path('/kaggle/input'); stage=Path('/kaggle/temp/srq_m14_sources'); stage.mkdir(parents=True,exist_ok=True)
def unique_preserved(name):
    matches=sorted(path for candidate in (name,name+'.bin') for path in INPUT_ROOT.rglob(candidate) if path.is_file())
    assert len(matches)==1,f'Upload exactly one byte-preserved {name}.bin; found {matches}'
    return matches[0]
SOURCE_PATHS={}
for key,name in SOURCE_NAMES.items():
    source=unique_preserved(name); assert sha_raw(source)==SOURCE_SHA[key],(source,sha_raw(source),SOURCE_SHA[key])
    destination=stage/name; shutil.copyfile(source,destination); SOURCE_PATHS[key]=str(destination)
cifar_dirs=sorted({p.parent for p in INPUT_ROOT.rglob('meta') if p.is_file() and (p.parent/'train').is_file() and (p.parent/'test').is_file()})
assert len(cifar_dirs)==1,f'Attach zaphat206/cifar-100 exactly once; found {cifar_dirs}'
CIFAR_ROOT=str(cifar_dirs[0])
candidates=[p for p in INPUT_ROOT.rglob('model.safetensors') if p.is_file() and p.stat().st_size==CHECKPOINT_SIZE and sha_raw(p)==CHECKPOINT_SHA]
if candidates: assert len(candidates)==1; CHECKPOINT_PATH=str(candidates[0])
else:
    from huggingface_hub import hf_hub_download
    CHECKPOINT_PATH=hf_hub_download(repo_id='timm/vit_base_patch16_224.augreg2_in21k_ft_in1k',filename='model.safetensors')
assert Path(CHECKPOINT_PATH).stat().st_size==CHECKPOINT_SIZE and sha_raw(CHECKPOINT_PATH)==CHECKPOINT_SHA
print('M6 + M11 + M13-N + CIFAR + CHECKPOINT: PASS')

In [ ]:
# Focused gates, then TRAIN-only feature extraction.
subprocess.run([sys.executable,'-B','-m','pytest','-q','-p','no:cacheprovider','tests/test_srq_generalization_m14.py','tests/test_srq_generalization_m13n.py','tests/test_loranpac_analytic_frontend.py','tests/test_analytic_ridge_backend.py'],check=True)
cache=Path(FEATURE_CACHE_DIR)
if not (cache/'train.pt').is_file():
    command=[sys.executable,'-u','tools/experiment_runner.py','--extract-features-only','--extract-train-only','--root',CIFAR_ROOT,'--backbone-checkpoint',CHECKPOINT_PATH,'--backbone-checkpoint-size',str(CHECKPOINT_SIZE),'--backbone-checkpoint-sha256',CHECKPOINT_SHA,'--feature-cache-dir',FEATURE_CACHE_DIR,'--output-dir','/kaggle/working/unused_m14','--dataset','CIFAR-100','--model-name','vit_base_patch16_224','--data-augmentation','vit','--seed','2025','--num-classes','100','--num-tasks','10','--device','cuda','--batch-size',str(BATCH_SIZE),'--num-workers',str(NUM_WORKERS)]
    subprocess.run(command,check=True)
assert (cache/'train.pt').is_file() and not (cache/'test.pt').exists()
print('M14 PREFLIGHT + TRAIN CACHE: PASS; test.pt ABSENT')

In [ ]:
# Long, atomic-unit, resumable run. Accuracy is never a completion gate.
unit_dir=Path(OUTPUT_DIR)/'units'; unit_dir.mkdir(parents=True,exist_ok=True)
prior={}
for candidate in INPUT_ROOT.rglob('s*_w*.json'):
    if candidate.is_file(): prior.setdefault(candidate.name,[]).append(candidate)
for name,candidates in prior.items():
    digests={sha_raw(path) for path in candidates}; assert len(digests)==1,f'Conflicting prior M14 unit {name}: {candidates}'
    destination=unit_dir/name
    if not destination.exists(): shutil.copyfile(candidates[0],destination)
print('PRIOR M14 UNITS IMPORTED:',len(list(unit_dir.glob('*.json'))))
command=[sys.executable,'-u',RUNNER,'run','--config',CONFIG,'--source-m6-artifact',SOURCE_PATHS['m6'],'--source-m11-artifact',SOURCE_PATHS['m11'],'--source-m13n-artifact',SOURCE_PATHS['m13n'],'--feature-cache-dir',FEATURE_CACHE_DIR,'--output-dir',OUTPUT_DIR,'--device','cuda','--require-clean-git']
print('M14 START: 6 seeds x 2 widths x 5 methods = 60 units.',flush=True)
completed=subprocess.run(command); RUN_RETURN_CODE=completed.returncode
result_path=Path(OUTPUT_DIR)/'m14_results.json'
if not result_path.is_file():
    completed_units=sorted((Path(OUTPUT_DIR)/'units').glob('*.json'))
    print('COMPLETED UNIT COUNT:',len(completed_units)); print([p.name for p in completed_units])
    raise RuntimeError('M14 stopped inside a unit. Re-run this cell in the same session; completed units will be reused.')
result=json.loads(result_path.read_text())
print('STATUS:',result['status']); print('SUMMARY:',json.dumps(result['summary'],indent=2)); print('GATES:',json.dumps(result['gates'],indent=2))

In [ ]:
# Paper-ready aggregate table and vector accuracy/state plot.
import pandas as pd, matplotlib.pyplot as plt
rows=[]
for item in result['aggregate']:
    rows.append({'width':item['width'],'method':item['method'],'reference':item['paired_reference'],'AIA_mean':item['validation_aia_percent']['mean'],'AIA_sd':item['validation_aia_percent']['sample_standard_deviation'],'Final_mean':item['final_validation_accuracy_percent']['mean'],'Final_sd':item['final_validation_accuracy_percent']['sample_standard_deviation'],'delta_AIA_mean':item['paired_delta_aia_percent']['mean'],'positive_seeds':item['positive_delta_aia_seed_count'],'state_MiB':item['final_total_persistent_bytes']['mean']/2**20,'update_s':item['analytic_update_seconds']['mean']})
frame=pd.DataFrame(rows); display(frame)
fig,axes=plt.subplots(1,2,figsize=(11,4.2))
for width,part in frame.groupby('width'):
    axes[0].scatter(part.state_MiB,part.AIA_mean,label=f'{width//1000}k')
    for _,row in part.iterrows(): axes[0].annotate(row['method'],(row.state_MiB,row.AIA_mean),fontsize=6)
low=frame[frame.method.str.startswith('loranpac')]
axes[1].bar([f"{r.width//1000}k\n{r.method.replace('loranpac_','')}" for _,r in low.iterrows()],low.delta_AIA_mean)
axes[0].set_xlabel('Persistent state (MiB)'); axes[0].set_ylabel('Validation AIA (%)'); axes[1].set_ylabel('LoRanPAC - matched SRQ AIA (pp)')
for ax in axes: ax.grid(alpha=.25); ax.tick_params(axis='x',labelsize=7)
axes[0].legend(); fig.tight_layout(); plot=Path(OUTPUT_DIR)/'m14_multiseed_equal_byte.svg'; fig.savefig(plot,format='svg'); plt.show(); plt.close(fig)

In [ ]:
# Export compact auditable evidence; source ZIPs and feature caches are excluded.
export=Path(EXPORT_PATH)
members=[Path(OUTPUT_DIR)/'m14_results.json',Path(OUTPUT_DIR)/'m14_aggregate.csv',Path(OUTPUT_DIR)/'m14_multiseed_equal_byte.svg',Path(CONFIG),Path('docs/research/SRQ_GENERALIZATION_M14_PLAN.md')]
members.extend(sorted((Path(OUTPUT_DIR)/'units').glob('*.json')))
manifest={}
with zipfile.ZipFile(export,'w',compression=zipfile.ZIP_DEFLATED) as archive:
    for path in members: archive.write(path,path.name); manifest[path.name]=sha_raw(path)
    archive.writestr('MANIFEST.json',json.dumps({'schema_version':1,'artifact':'M14 LoRanPAC multi-seed train-only','files':manifest},indent=2)+'\n')
assert export.is_file() and zipfile.is_zipfile(export)
print('FINAL ARTIFACT:',export,'SHA-256:',sha_raw(export),'bytes:',export.stat().st_size)
from IPython.display import FileLink,display
display(FileLink(str(export)))
assert RUN_RETURN_CODE==0 and result['status']=='PASS_M14_LORANPAC_MULTISEED_TRAIN_ONLY','Preserve the artifact and gates; do not retry or change thresholds from accuracy.'